# 03 — Compare classifier families (`src/model_selection.py`)

Tunes LSTM, 1D CNN, gradient boosting and random forest with the expanding-window
cross-validation protocol in `src/model_selection.py`, then picks the best by
cross-validated F1. This is the slower, heavier notebook (a full grid search plus
5-seed ensembles for the two neural models), so it benefits the most from not
tying up your own Mac for however long it takes.

## Important: this script pins itself to the CPU, on purpose

`src/model_selection.py` calls `tf.config.set_visible_devices([], "GPU")` at the
top of `main()`, with a comment explaining why: for networks this small, the
overhead of dispatching to a GPU was found to be slower than just running on
CPU, at least on your Mac's Metal backend. That means **setting a GPU runtime for
this notebook has no effect unless you change that line**, Colab's GPU will sit
idle and everything still runs on CPU, same as it does locally.

Leave it as-is for a first run (keeps the result directly comparable to your
local runs). If you want to test whether Colab's GPU actually changes that
finding, comment out the `set_visible_devices` line in `src/model_selection.py`
after cloning, before running the cell below, and note in your results that this
run used a GPU so it isn't silently compared as if it were the CPU baseline.

In [ ]:
# --- One-time per Colab session: clone the repo and install dependencies ---
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")  # set once: key icon in the left sidebar, see colab/README.md
REPO = "DaniellaWambua/Real-Time-Flood-Data-Pipeline"
BRANCH = "colab-training"
REPO_DIR = "/content/repo"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git pull

!pip install -q -r requirements-colab.txt


## Run the comparison

`--models` can narrow this to a subset (for example just `LSTM` to re-run one
model while keeping the other three's saved results), see the script's own
docstring. Leaving it default runs all four plus the LSTM+Random Forest
combination.

In [ ]:
# Uncomment to force a specific dataset version instead of config.py's default:
# import os
# os.environ["FLOOD_DATASET"] = "v7"

!python -m src.model_selection
# !python -m src.model_selection --models LSTM   # to re-run just one model


## Push the results back

This writes per-model `.keras` / `.joblib` files under
`models/flood_classifiers/<dataset version>/`, plus JSON results and a
`summary.json` under `reports/model_selection/<dataset version>/`.

In [ ]:
!git add models/flood_classifiers/ reports/model_selection/
!git -c user.email="youngkyper254@gmail.com" -c user.name="Daniella Wambua" \
     commit -m "Run classifier comparison (src/model_selection.py) on Colab"
!git push origin colab-training


## Back in VS Code

```bash
git checkout colab-training
git pull
```

`reports/model_selection/<version>/summary.json` has the winning model and the
full comparison table.